# Notebook 1: Data Collection

## 1.0 Preamble

This notebook orchestrates expert driving data collection across the CARLA simulator. An autopilot-driven vehicle is placed in every combination of weather, time-of-day, and traffic density for each of six towns, producing a diverse dataset for imitation learning. The pipeline saves raw camera frames, vehicle state, expert actions, and environmental metadata as compressed `.npz` chunk files. Running the full grid yields approximately 97,200 frames -- enough to train a robust behavior-cloning model that generalizes across conditions. Each town must be collected in a separate CARLA session due to a hardware constraint that prevents runtime map switching on the RTX 5080.

In [ ]:
import sys
import os
import json
import itertools
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from tqdm.auto import tqdm

# Ensure project root is on sys.path so src imports resolve
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import carla
from src.config import load_config
from src.agents import DataCollectionAgent
from src.carla_env import CarlaEnv

# Plotting defaults
plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (10, 5)})

DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Data is collected for every (sensor suite x town) combination and stored
# under data/{suite}/{town}/. These two lists drive both the collection loop
# (1.4) and every validation/analysis cell below, so they live here in setup.
SENSOR_SUITES = ["single_cam", "multi_cam", "lidar"]
TOWNS = ["Town01", "Town02", "Town03", "Town04", "Town05", "Town10HD"]

## 1.1 Configuration

All parameters governing the data collection process -- camera resolution, weather presets, traffic densities, and frame targets -- live in a single YAML file (`configs/collection.yaml`). Loading them through `load_config()` ensures that notebooks, agents, and scripts always operate with the same values. This eliminates the risk of inconsistent hardcoded numbers drifting apart as the project evolves.

### 1.1.1 Loading Parameters

We load the collection config and print its key parameters so the reader can verify the settings before committing to a multi-hour collection run. The config specifies 300 frames per condition, 800x600 camera resolution, and six weather presets. These values match the requirements defined in `configs/collection.yaml` and are the single source of truth for all collection code.

In [2]:
cfg = load_config("collection")

print("=== Collection Configuration ===")
print(f"  Seed:                 {cfg['seed']}")
print(f"  Image resolution:     {cfg['image_width']}x{cfg['image_height']}")
print(f"  Frames per condition: {cfg['frames_per_condition']}")
print(f"  Chunk size:           {cfg['chunk_size']}")
print(f"  Weather presets:      {cfg['weather_presets']}")
print(f"  TOD sun angles:       {cfg['tod_sun_angles']}")
print(f"  Traffic densities:    {cfg['traffic_vehicle_counts']}")

=== Collection Configuration ===
  Seed:                 42
  Image resolution:     800x600
  Frames per condition: 300
  Chunk size:           2400
  Weather presets:      ['ClearNoon', 'CloudyNoon', 'WetNoon', 'HardRainNoon', 'ClearSunset', 'ClearNight']
  TOD sun angles:       {'day': 75.0, 'sunset': 15.0, 'night': -90.0}
  Traffic densities:    {'low': 10, 'medium': 30, 'high': 60}


### 1.1.2 Condition Grid

The full experimental design is a factorial grid of 6 weather presets, 3 times of day, 3 traffic densities, and 6 towns, giving 6 x 6 x 3 x 3 = 324 unique conditions. Each town contributes 54 conditions (6 weathers x 3 TODs x 3 traffic levels). At 300 frames per condition the dataset targets 97,200 total frames. Building the grid as a DataFrame makes it easy to audit coverage and spot any missing cells after collection.

In [3]:
weathers = cfg["weather_presets"]
tods = list(cfg["tod_sun_angles"].keys())
traffics = list(cfg["traffic_vehicle_counts"].keys())

condition_grid = pd.DataFrame(
    list(itertools.product(weathers, tods, traffics, TOWNS)),
    columns=["weather", "tod", "traffic", "town"],
)

print(f"Condition grid shape: {condition_grid.shape}  (expected 324 rows x 4 columns)")
print(f"\nConditions per town: {len(condition_grid) // len(TOWNS)}")
print(f"Total frames target: {len(condition_grid) * cfg['frames_per_condition']:,}")
print()
condition_grid.head(10)

Condition grid shape: (324, 4)  (expected 324 rows x 4 columns)

Conditions per town: 54
Total frames target: 97,200



,weather,tod,traffic,town
0,ClearNoon,day,low,Town01
1,ClearNoon,day,low,Town02
2,ClearNoon,day,low,Town03
3,ClearNoon,day,low,Town04
4,ClearNoon,day,low,Town05
5,ClearNoon,day,low,Town10HD
6,ClearNoon,day,medium,Town01
7,ClearNoon,day,medium,Town02
8,ClearNoon,day,medium,Town03
9,ClearNoon,day,medium,Town04


## 1.2 Environment Setup

Before starting collection we verify that the CARLA server is reachable and correctly configured. A failed connection here is far cheaper to debug than discovering it mid-collection after an hour of waiting. We also confirm synchronous mode and sensor settings so the simulation advances deterministically one tick at a time.

### 1.2.1 CARLA Connection Verification

This cell attempts to connect to the CARLA server on localhost:2000 and prints the server and client API versions. If the versions do not match, sensor data may be malformed or missing entirely. The connection must succeed before any data can be collected. A timeout of 10 seconds avoids hanging indefinitely if the server is not running.

In [4]:
client = carla.Client("localhost", 2000)
client.set_timeout(10.0)

try:
    server_version = client.get_server_version()
    client_version = client.get_client_version()
    print(f"CARLA server version: {server_version}")
    print(f"CARLA client version: {client_version}")
    if server_version != client_version:
        print("WARNING: version mismatch -- sensor data may be unreliable.")
    else:
        print("Versions match. Connection OK.")
except RuntimeError as exc:
    print(f"ERROR: Cannot reach CARLA server -- {exc}")
    print("Launch CARLA first:  scripts/launch_carla.bat")
    raise

CARLA server version: 0.9.16
CARLA client version: 0.9.16
Versions match. Connection OK.


### 1.2.2 Sanity Check

We verify the CARLA server is reachable and in a safe configuration before starting a multi-hour collection run. Two things are checked and corrected if needed:

**Synchronous mode guard.** `CarlaEnv` runs CARLA in synchronous mode during collection — the client calls `world.tick()` to advance the simulation one frame at a time. If a prior session crashed without calling `env.close()`, the server remains in synchronous mode with no one feeding ticks. Calling `world.tick()` from the kernel in that state claims frame authority and leaves the server waiting indefinitely for the next tick; any subsequent CARLA call hangs and the kernel dies silently (`0xC0000409` on Windows). This cell detects and resets stale synchronous mode before touching anything else.

**`get_snapshot()` instead of `tick()`.** The check uses `world.get_snapshot()` rather than `world.tick()`. Both confirm the server is alive, but `get_snapshot()` is purely observational — it reads the current frame without advancing the simulation or claiming frame authority. The cell is safe to re-run at any time.

**Client teardown.** After the check, `del client` releases the Boost.Asio connection. The subprocess in section 1.4.1 creates its own independent connection; stacking two live clients on port 2000 across a kernel restart causes silent connection errors.

In [ ]:
world = client.get_world()
settings = world.get_settings()

print(f"Current map:          {world.get_map().name}")
print(f"Synchronous mode:     {settings.synchronous_mode}")
print(f"Fixed delta seconds:  {settings.fixed_delta_seconds}")
if settings.fixed_delta_seconds:
    print(f"Effective FPS:        {1.0 / settings.fixed_delta_seconds:.0f}")
else:
    print( "Effective FPS:        Variable (async)")

# If a prior CarlaEnv session crashed without calling close(), the server may
# still be in synchronous mode. In sync mode, world.tick() here would make this
# kernel the frame authority — with nothing feeding subsequent ticks the server
# hangs and the kernel dies silently (0xC0000409 on Windows/RTX 5080).
if settings.synchronous_mode:
    print("\nWARNING: server left in synchronous mode by a prior crashed session.")
    print("Resetting to asynchronous mode ...")
    settings.synchronous_mode = False
    settings.fixed_delta_seconds = None
    world.apply_settings(settings)
    print("Reset OK — server is now in async mode.")

# get_snapshot() instead of tick(): both confirm the server is alive, but
# get_snapshot() is purely observational — it never claims frame authority,
# so this cell is safe to re-run at any time without side effects.
snapshot = world.get_snapshot()
print(f"\nSanity check OK — frame: {snapshot.frame}, "
      f"elapsed: {snapshot.timestamp.elapsed_seconds:.1f}s")

print(f"\nExpected sensor config:")
print(f"  Camera resolution: {cfg['image_width']}x{cfg['image_height']}")
print( "  Sensors: RGB camera, collision, lane invasion")

# Release the client so its Boost.Asio threads don't accumulate on re-run or
# kernel restart. The collect_one_town.py subprocess manages its own connection.
del client, world, settings
print("\nClient released. Ready for collection (section 1.4.1).")

## 1.3 Data Schema

Understanding the exact layout of the saved data is essential for downstream consumers (preprocessing, training, evaluation). Each chunk file is a compressed NumPy archive containing synchronized arrays of camera images, vehicle states, expert actions, and environmental metadata. This section documents every field so that anyone loading a chunk knows exactly what they are working with.

### 1.3.1 Frame Schema

Each `.npz` chunk file stores up to `chunk_size` frames with the fields listed below. The sensor payload field differs by suite (see the next table); all other fields are identical across suites. States record the vehicle's instantaneous speed, heading, speed limit, lane count, and junction status. Actions record the autopilot's steering, throttle, and brake commands -- the ground truth labels for imitation learning. The style field is always "standard" during expert collection; PPO fine-tuning learns to vary style through reward shaping.

**Sensor payload field (one per chunk, depends on the suite that collected it):**

| Suite | Field | dtype | Shape (per chunk) | Description |
|---|---|---|---|---|
| `single_cam` | `images` | uint8 | (N, 600, 800, 3) | Front RGB camera frame |
| `multi_cam` | `images_multi` | uint8 | (N, 3, 600, 800, 3) | [front, left, right] RGB frames |
| `lidar` | `bev` | float32 | (N, 100, 200, 3) | BEV projection: [height, density, intensity] |

**Common fields (identical across all suites):**

| Field | dtype | Shape (per chunk) | Unit / Description |
|---|---|---|---|
| `states` | float32 | (N, 5) | [speed_kmh, heading_deg, speed_limit_kmh, lane_count, is_junction] |
| `actions` | float32 | (N, 3) | [steer, throttle, brake] from autopilot |
| `locations` | float32 | (N, 2) | [x, y] world coordinates (meters) |
| `tl_states` | uint8 | (N,) | Traffic light state: 0=Red, 1=Yellow, 2=Green, 3=Off |
| `speed_limits` | float32 | (N,) | Posted speed limit (km/h) |
| `weather_preset` | str | (N,) | Weather preset name (e.g. ClearNoon) |
| `road_type` | str | (N,) | highway, rural, or urban |
| `time_of_day` | str | (N,) | day, sunset, or night |
| `traffic_density` | str | (N,) | low, medium, or high |
| `style` | str | (N,) | Always "standard" during expert collection |

### 1.3.2 Lane Change Event Schema

In addition to the per-frame data, the collection agent records collision events in a separate `collision_log.npz` file per town. Each collision triggers an episode reset, so these events also mark episode boundaries in the data. The log enables post-hoc analysis of collision frequency under different conditions and helps identify problematic road segments.

| Field | dtype | Description |
|---|---|---|
| `weather` | str | Weather preset active during the collision |
| `tod` | str | Time of day (day, sunset, night) |
| `traffic` | str | Traffic density tier (low, medium, high) |
| `frame` | int | Frame index within the condition when collision occurred |

## 1.4 Collection Loop

Data collection runs every **sensor suite** (`single_cam`, `multi_cam`, `lidar`) across every **town**, one combination at a time. Each combination is collected in its own subprocess that owns the full CARLA lifecycle — it launches the server, loads the target map via `client.load_world()`, collects that town's 54 conditions (6 weathers x 3 times of day x 3 traffic densities), then kills the server before exiting.

Collection is organized per-town because CARLA cannot switch maps *in place after a sensor cycle* on this hardware (RTX 5080 Blackwell). The first `load_world()` on a freshly launched server is fine; reusing a server across towns is what aborts. Spawning a fresh server per town sidesteps this entirely. The agent automates map loading, weather changes, NPC spawning, autopilot driving, and chunk persistence.

Output is namespaced by suite: `data/{suite}/{town}/chunk_*.npz` (e.g. `data/lidar/Town03/chunk_0000.npz`). Legacy flat single-cam data under `data/{town}/` is left untouched.

### 1.4.1 Subprocess-Per-Combination Collection

Each (sensor suite, town) combination is collected by invoking `scripts/collect_one_town.py` as a separate subprocess. This isolation is required because libcarla's Boost.Asio streaming threads outlive the Python client object. If all combinations ran in the same process (the Jupyter kernel), stale background threads from a completed run would abort the next CARLA session and crash the kernel. Running each combination in its own subprocess lets the OS reap those threads cleanly on exit.

**The subprocess handles the full CARLA lifecycle for you** — it launches the CARLA server, loads the correct map, collects all 54 conditions, and shuts CARLA down before exiting. Do not start CARLA manually before running the cell below.

The loop is `for suite in SENSOR_SUITES: for town in TOWNS`, so it produces 3 suites x 6 towns = 18 subprocess runs. Each run takes approximately 1–2 hours. Exit code `0` means all chunks were saved; exit code `3` means collection completed but fewer frames than expected were captured (still usable — check the validation section). Any other exit code indicates a failure for that combination, and the loop continues to the next one.

In [ ]:
import subprocess
import sys

# Collect every (sensor suite x town) combination. Each combination runs in
# its own subprocess so libcarla's Boost.Asio streaming threads are reaped by
# the OS on exit -- never reuse a CARLA session across towns or suites.
# SENSOR_SUITES and TOWNS are defined once in the setup cell (1.0).
SCRIPT = PROJECT_ROOT / "scripts" / "collect_one_town.py"

# results[suite][town] = number of chunks saved
results = {suite: {} for suite in SENSOR_SUITES}

for suite in SENSOR_SUITES:
    print(f"\n{'#'*60}")
    print(f"# SENSOR SUITE: {suite}")
    print(f"{'#'*60}")

    for town in TOWNS:
        print(f"\n{'='*60}")
        print(f"Collecting {suite} / {town} ...")
        print(f"{'='*60}")

        # The subprocess owns the full CARLA lifecycle: launch -> collect 54
        # conditions -> kill CARLA -> os._exit(0). The notebook kernel never
        # holds a CARLA client, so it cannot be destabilized by collection.
        proc = subprocess.run(
            [sys.executable, str(SCRIPT), "--town", town,
             "--sensor-suite", suite,
             "--data-dir", str(DATA_DIR)],
            cwd=str(PROJECT_ROOT),
        )

        suite_town_dir = DATA_DIR / suite / town
        if proc.returncode in (0, 3):
            chunks = sorted(suite_town_dir.glob("chunk_*.npz"))
            results[suite][town] = len(chunks)
            print(f"{suite}/{town}: {len(chunks)} chunks saved "
                  f"(exit {proc.returncode})")
        else:
            results[suite][town] = 0
            print(f"WARNING: {suite}/{town} exited with code {proc.returncode}")

print("\n=== Collection Summary ===")
for suite in SENSOR_SUITES:
    print(f"\n[{suite}]")
    for town, n in results[suite].items():
        status = "OK" if n > 0 else "FAILED"
        print(f"  {town}: {n} chunks  [{status}]")

In [ ]:
expected_per_town = 54 * cfg["frames_per_condition"]  # 54 * 300 = 16,200

# The per-suite npz payload key differs: single_cam -> "images",
# multi_cam -> "images_multi", lidar -> "bev". A chunk's frame count is the
# length of whichever payload key is present.
_PAYLOAD_KEYS = ("images", "images_multi", "bev")


def _chunk_frame_count(data) -> int:
    for key in _PAYLOAD_KEYS:
        if key in data:
            return data[key].shape[0]
    return 0


# frame_counts[suite][town], chunk_counts[suite][town]
frame_counts = {suite: {} for suite in SENSOR_SUITES}
chunk_counts = {suite: {} for suite in SENSOR_SUITES}

for suite in SENSOR_SUITES:
    for town in TOWNS:
        town_dir = DATA_DIR / suite / town
        chunks = sorted(town_dir.glob("chunk_*.npz")) if town_dir.exists() else []
        chunk_counts[suite][town] = len(chunks)
        total = 0
        for chunk_path in chunks:
            with np.load(chunk_path, allow_pickle=True) as data:
                total += _chunk_frame_count(data)
        frame_counts[suite][town] = total

print(f"Expected per town: {expected_per_town:,}")
for suite in SENSOR_SUITES:
    print(f"\n[{suite}]")
    print(f"{'Town':<12} {'Frames':>8} {'Chunks':>8} {'Status'}")
    print("-" * 45)
    for town in TOWNS:
        status = "OK" if frame_counts[suite][town] >= expected_per_town * 0.95 else "LOW"
        print(f"{town:<12} {frame_counts[suite][town]:>8,} "
              f"{chunk_counts[suite][town]:>8} {status}")

total_frames = sum(
    frame_counts[s][t] for s in SENSOR_SUITES for t in TOWNS
)
print(f"\nTotal frames across all suites and towns: {total_frames:,}")

In [ ]:
tod_label_counts = {"day": 0, "sunset": 0, "night": 0}
collision_counts = {}  # collision_counts[town] summed across suites

# Metadata (time_of_day, collisions) is sensor-independent, so we aggregate
# across all suites. Iterate every suite/town dir that exists.
for town in TOWNS:
    collision_counts[town] = 0
    for suite in SENSOR_SUITES:
        town_dir = DATA_DIR / suite / town
        chunks = sorted(town_dir.glob("chunk_*.npz")) if town_dir.exists() else []
        for chunk_path in chunks:
            with np.load(chunk_path, allow_pickle=True) as data:
                tods_arr = data["time_of_day"].astype(str)
                for t in tods_arr:
                    if t in tod_label_counts:
                        tod_label_counts[t] += 1

        # Collision log (one per suite/town)
        coll_path = town_dir / "collision_log.npz"
        if coll_path.exists():
            with np.load(coll_path, allow_pickle=True) as clog:
                collision_counts[town] += len(clog["frame"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Time-of-day distribution
axes[0].bar(tod_label_counts.keys(), tod_label_counts.values(),
            color=["#FFC107", "#FF5722", "#3F51B5"])
axes[0].set_title("Frame Count by Time of Day (all suites)")
axes[0].set_ylabel("Frames")
axes[0].yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

# Collision frequency per town
axes[1].bar(collision_counts.keys(), collision_counts.values(),
            color="#E53935")
axes[1].set_title("Collision Events per Town (all suites)")
axes[1].set_ylabel("Collisions")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

### 1.6.1 Aggregate Statistics Table

This table summarizes the total frame count, estimated recording duration (at 20 FPS), and approximate disk usage for each town. The total across all towns should be close to 97,200 frames. Disk size is computed from chunk file sizes on disk for accuracy.

### 1.6.2 Condition Coverage Heatmap

This heatmap cross-tabulates weather presets against time-of-day and shows the total frame count in each cell (summed across all towns and traffic densities). A uniform heatmap confirms balanced coverage. Any dark or zero cells indicate missing conditions that need to be re-collected.

In [ ]:
# Build a weather x tod frame count matrix across all data (all suites)
weather_tod_counts = {w: {t: 0 for t in tods} for w in weathers}

for town in TOWNS:
    for suite in SENSOR_SUITES:
        town_dir = DATA_DIR / suite / town
        chunks = sorted(town_dir.glob("chunk_*.npz")) if town_dir.exists() else []
        for chunk_path in chunks:
            with np.load(chunk_path, allow_pickle=True) as data:
                w_arr = data["weather_preset"].astype(str)
                t_arr = data["time_of_day"].astype(str)
                for w, t in zip(w_arr, t_arr):
                    if w in weather_tod_counts and t in weather_tod_counts[w]:
                        weather_tod_counts[w][t] += 1

heatmap_df = pd.DataFrame(weather_tod_counts).T
heatmap_df = heatmap_df[tods]  # enforce column order: day, sunset, night

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(heatmap_df.values, cmap="YlOrRd", aspect="auto")
ax.set_xticks(range(len(tods)))
ax.set_xticklabels(tods)
ax.set_yticks(range(len(weathers)))
ax.set_yticklabels(weathers)
ax.set_xlabel("Time of Day")
ax.set_ylabel("Weather Preset")
ax.set_title("Frame Count: Weather x Time of Day (all suites)")

# Annotate cells
for row in range(len(weathers)):
    for col in range(len(tods)):
        val = heatmap_df.values[row, col]
        ax.text(col, row, f"{val:,}", ha="center", va="center",
                fontsize=9,
                color="white" if val > heatmap_df.values.max() * 0.6
                else "black")

plt.colorbar(im, ax=ax, label="Frame Count")
plt.tight_layout()
plt.show()

## 1.7 Save and Export

The final step is to write a machine-readable index of the collected dataset and a snapshot of the collection configuration. The JSON index enables downstream code to discover all available chunks without scanning the filesystem, and the config snapshot records exactly which parameters were used for this collection run.

### 1.7.1 Write Dataset Index

This cell generates a JSON file listing every chunk file per town, along with its frame count. The index file makes it straightforward for training code to enumerate available data without filesystem globbing, and it serves as a manifest for data integrity checks.

In [ ]:
dataset_index = {}  # dataset_index[suite][town] = [{file, frames}, ...]

for suite in SENSOR_SUITES:
    dataset_index[suite] = {}
    for town in TOWNS:
        town_dir = DATA_DIR / suite / town
        chunks = sorted(town_dir.glob("chunk_*.npz")) if town_dir.exists() else []
        town_entries = []
        for chunk_path in chunks:
            with np.load(chunk_path, allow_pickle=True) as data:
                n = _chunk_frame_count(data)  # suite-aware (defined in 1.5)
            town_entries.append({
                "file": str(chunk_path.relative_to(PROJECT_ROOT)),
                "frames": int(n),
            })
        dataset_index[suite][town] = town_entries

index_path = RESULTS_DIR / "dataset_index.json"
with open(index_path, "w") as f:
    json.dump(dataset_index, f, indent=2)

print(f"Dataset index written to: {index_path}")
print(f"Suites indexed: {list(dataset_index.keys())}")
total_indexed = sum(
    e["frames"]
    for suite_entries in dataset_index.values()
    for entries in suite_entries.values()
    for e in entries
)
print(f"Total indexed frames: {total_indexed:,}")

### 1.7.2 Serialize Config Snapshot

Saving the exact collection configuration as a JSON snapshot ensures full reproducibility. If questions arise later about which parameters were used during data collection, this file provides the definitive answer. It is timestamped implicitly by the file's modification date.

In [ ]:
config_snapshot_path = RESULTS_DIR / "collection_config_snapshot.json"
with open(config_snapshot_path, "w") as f:
    json.dump(cfg, f, indent=2, default=str)

print(f"Config snapshot saved to: {config_snapshot_path}")
print(f"Keys: {list(cfg.keys())}")